In [4]:
import ollama

In [5]:
inventory_db = {
    "apple": {"price": 1.0, "quantity": 100},
    "banana": {"price": 0.5, "quantity": 150},
    "orange": {"price": 0.8, "quantity": 120}
}

In [6]:
# Tool 1: Check inventory
def check_inventory(item):
    item_name = item.lower()
    if item_name in inventory_db:
        return f"{item.capitalize()} - Price: ${inventory_db[item_name]['price']}, Quantity: {inventory_db[item_name]['quantity']}"
    else:
        return f"{item.capitalize()} is not available in the inventory."
    
# Tool 2: Business Logic for discounts for loyal customers
def calculate_loyality_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.25)  # Max discount of 25%
    discounted_price = base_price * (1 - discount)
    return discounted_price


In [7]:
available_tools = {
    "check_inventory": check_inventory,
    "calculate_loyality_discount": calculate_loyality_discount
}

In [20]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check the price and quantity of an item in the inventory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {"type": "string", "description": "Name of the item to check in inventory."}
                },
                "required": ["item"]
            }
        }
    }, 
    {
        "type": "function",
        "function": {
            "name": "calculate_loyality_discount",
            "description": "Calculate the discount for loyal customers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "float", "description": "The original price of the item."},
                    "years_as_customer": {"type": "integer", "description": "Number of years the customer has been with the business."}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [9]:
messages = [
    {"role": "user", "content": "I wants to buy the apple, how much it cost?"},
]

response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Model Response:", response)

Model Response: model='llama3.1:8b' created_at='2026-05-23T12:12:17.2145894Z' done=True done_reason='stop' total_duration=40117491700 load_duration=329930000 prompt_eval_count=249 prompt_eval_duration=34753493200 eval_count=17 eval_duration=4972158600 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'item': 'apple'}))]) logprobs=None


In [10]:
message = response["message"]
print("Model Message:", message)

Model Message: role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'item': 'apple'}))]


In [12]:
tool_calls = response["message"].get("tool_calls", [])
if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]
        
        if tool_name in available_tools:
            result = available_tools[tool_name](**tool_args)
            print(f"Tool '{tool_name}' called with arguments {tool_args} returned: {result}")

            messages.append(message)
            messages.append({"role": "tool", "content": result})
        else:
            print(f"Tool '{tool_name}' is not available.")



Tool 'check_inventory' called with arguments {'item': 'apple'} returned: Apple - Price: $1.0, Quantity: 100


In [13]:
final_response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Final Model Response:", final_response)

Final Model Response: model='llama3.1:8b' created_at='2026-05-23T12:13:00.2171355Z' done=True done_reason='stop' total_duration=24727668100 load_duration=345955200 prompt_eval_count=147 prompt_eval_duration=18259388700 eval_count=22 eval_duration=6054517400 message=Message(role='assistant', content='The price of an apple is $1.00, and there are currently 100 apples in stock.', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [14]:
print(final_response["message"]["content"])

The price of an apple is $1.00, and there are currently 100 apples in stock.


In [22]:
prompt = "I am a loyal customer for 5 years, how much discount I can get on the apple?"

In [23]:
messages.append({"role": "user", "content": prompt})
response = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Response message:", response["message"])


ValidationError: 1 validation error for Message
content
  Input should be a valid string [type=string_type, input_value=0.75, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [17]:
tool_calls = response["message"].get("tool_calls", [])
if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]
        
        if tool_name in available_tools:
            result = available_tools[tool_name](**tool_args)
            print(f"Tool '{tool_name}' called with arguments {tool_args} returned: {result}")

            messages.append(message)
            messages.append({"role": "tool", "content": result})
        else:
            print(f"Tool '{tool_name}' is not available.")

Tool 'calculate_loyality_discount' called with arguments {'years_as_customer': 5, 'base_price': 1} returned: 0.75


In [19]:
final_response_with_discount = ollama.chat(model="llama3.1:8b", messages=messages, tools=tools)
print("Final Model Response with Discount:", final_response_with_discount["message"])

ValidationError: 1 validation error for Message
content
  Input should be a valid string [type=string_type, input_value=0.75, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type